# MiniFact：受控 FP16 实验
在新的 Colab GPU 会话中运行下面一个单元格。源码固定到已验证的提交，与本地运行相同；不覆盖旧 notebook。
独立环境会安装固定版本 PyTorch 和 Transformers，首次运行需要下载公共依赖／模型。结果只包含合成测试句和运行元数据，不读取 Drive 文件。
完成后下载 `minifact_controlled_colab.zip`。只有比较器通过一致性检查后才能报告跨环境比较。


In [ ]:
# Same source commit and inference entry point as the local FP16 run.
import os, sys, subprocess, tempfile, json, zipfile
from pathlib import Path

SOURCE_COMMIT = '8376eef7b7b8f7ba4836c9cabd381ae19a6680db'
REPOSITORY = "https://github.com/GU143312/minifact-llm.git"
runtime = Path("/content/minifact-runtime")
runtime.mkdir(exist_ok=True)
os.environ["UV_CACHE_DIR"] = str(runtime / "uv-cache")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HUB_DISABLE_XET"] = "1"

def checked(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)

work = Path(tempfile.mkdtemp(prefix="minifact-", dir="/content"))
repo = work / "repo"
checked(["git", "clone", "--quiet", REPOSITORY, repo])
checked(["git", "checkout", "--quiet", SOURCE_COMMIT], cwd=repo)
checked([sys.executable, "-m", "pip", "install", "--quiet", "uv"])
env = runtime / ".venv"
python = env / "bin" / "python"
if not python.exists():
    checked([sys.executable, "-m", "uv", "venv", "--python", sys.executable, env])
checked([sys.executable, "-m", "uv", "pip", "install", "--python", python, "-r", repo / "requirements.txt"])
checked([python, "-m", "unittest", "discover", "-s", "tests", "-v"], cwd=repo)
checked([python, "minifact.py", "--label", "colab", "--output", "reports/colab_fp16.json", "--download", "--gpu-memory", "5GiB"], cwd=repo)
checked([python, "evaluate.py", "reports/colab_fp16.json", "--output", "reports/colab_metrics.json"], cwd=repo)
bundle = Path("/content/minifact_controlled_colab.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["colab_fp16.json", "colab_metrics.json"]:
        z.write(repo / "reports" / name, name)
print("SOURCE_COMMIT:", SOURCE_COMMIT)
print("RESULT_BUNDLE:", bundle)
print(json.loads((repo / "reports/colab_metrics.json").read_text())["totals"])
from google.colab import files
files.download(str(bundle))
